In [1]:

import gc
import multiprocessing
import os
import re
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
from pyulog import ULog

In [2]:
# return the array of combined log file lines
def get_log_data(baseLogDir, iteration, testCase, model="iris"):
    #logPath = os.path.join(baseLogDir, iteration, model, testCase, "log-combined.log_plain.log")

    # 1) testCase 디렉토리 경로
    case_dir = os.path.join(baseLogDir, iteration, model, testCase)

    # 2) case_dir 안의 서브디렉토리(폴더) 목록 중 디렉토리인 것만 필터
    subdirs = [
        d for d in os.listdir(case_dir)
        if os.path.isdir(os.path.join(case_dir, d))
    ]

    # 3) 서브디렉토리가 정확히 하나라면, 그 이름을 extra_folder로 사용
    if len(subdirs) == 1:
        extra_folder = subdirs[0]
        log_path = os.path.join(case_dir, extra_folder, "log-combined.log_plain.log")
    else:
        # 서브폴더가 없거나 둘 이상일 경우, 예전 방식(바로 testCase 밑)을 시도
        log_path = os.path.join(case_dir, "log-combined.log_plain.log")

    if os.path.exists(log_path):
        with open(log_path, "r") as f:
            return f.readlines()
    return []

In [3]:
def parse_combined_log(log_lines):
    """
    combined-log 에서
      1) 정상 착륙 / 추락 여부 (landing_outcome)
      2) Gyro Bias injection 정보 (bias_info: dict with Time, x, y, z)
      3) ULog 파일명 (ulg_name)
    을 한 번의 순회로 추출하여 반환합니다.
    """
    landing_outcome = None
    bias_info = {}
    ulg_name = None
    ekf_detection = None

    # 패턴 준비
    size_pat = re.compile(r"Gyro Bias Updated\s*\{\s*x:\s*([-\d.]+),\s*y:\s*([-\d.]+),\s*z:\s*([-\d.]+)\s*\}")
    time_pat = re.compile(r"Gyro Bias Updated SimTime:\s*(\d+)")
    file_pat = re.compile(r"Opened full log file:\s+(.*\.ulg)")

    for line in log_lines:
        # 1) landing outcome
        if landing_outcome is None:
            if "정상 착륙" in line or "stable landing" in line:
                landing_outcome = "stable_landing"
            elif "Crash 감지" in line or "crash detected" in line:
                landing_outcome = "crash_detected"

        # 2) bias injection size/time
        if 'x' not in bias_info:
            m = size_pat.search(line)
            if m:
                bias_info['x'] = float(m.group(1))
                bias_info['y'] = float(m.group(2))
                bias_info['z'] = float(m.group(3))
        if 'Time' not in bias_info:
            m2 = time_pat.search(line)
            if m2:
                bias_info['Time'] = int(m2.group(1))

        # 3) ulg name
        if ulg_name is None:
            m3 = file_pat.search(line)
            if m3:
                ulg_name = m3.group(1)

        # 4) ekf gyro bias detection
        if ekf_detection is None:
            if "High Gyro Bias" in line:
                ekf_detection = True

        # 모두 찾았으면 루프 종료
        if landing_outcome and 'Time' in bias_info and ulg_name and ekf_detection:
            break

    # bias_info가 완전하지 않으면 None 처리
    if 'Time' not in bias_info or 'x' not in bias_info:
        bias_info = None

    return landing_outcome, bias_info, ulg_name, ekf_detection

In [4]:
# return the ulog parsed from the ulg file
def get_ulog(baseUlgDir, ulgFileName):
    normPath = os.path.normpath(ulgFileName)
    ulgPath = os.path.join(baseUlgDir, normPath)
    return ULog(ulgPath) if os.path.exists(ulgPath) else None

In [5]:
def find_stabilization_time(ulog, bias_time_us, window_s=20.0, threshold=1.2):
    """
    bias_time 이후 estimator_innovation_variances 의 gps_hpos[0] 이
    threshold 아래로 window_s 초(= window_s*1e6 us) 동안 연속 유지되는
    첫 시점을 반환. 못 찾으면 None.
    """
    data = pd.DataFrame(ulog.get_dataset("estimator_innovation_variances").data)
    df = data[data['timestamp'] >= bias_time_us].reset_index(drop=True)
    times = df['timestamp'].values
    vals = df['gps_hpos[0]'].values
    window_us = window_s * 1e6
    n = len(times)
    for i in range(n):
        if vals[i] < threshold:
            # 목표 종료 시각
            target = times[i] + window_us
            # 해당 시각 이상인 첫 인덱스 찾기
            j = (times >= target).argmax()
            if times[j] < target:
                # 전체 윈도우를 채우기엔 데이터가 부족
                continue
            # 중간에 threshold 위반이 없는지 확인
            if vals[i:j + 1].max() < threshold:
                return times[i]
    return None

In [6]:
def get_arm_time(ulog):
    armed_df = pd.DataFrame(ulog.get_dataset("actuator_armed").data)
    ev = armed_df[armed_df['armed'] == 1]
    if not ev.empty:
        return int(ev['timestamp'].iloc[0])
    return None

In [7]:
def get_stab_time(ulog, bias_info):
    if bias_info:
        return find_stabilization_time(ulog, bias_info['Time'])
    return None

In [8]:
def get_trimmed_topics(ulog, bias_info, stab_time, topics):
    """
    주어진 시간 구간 [start_us, end_us] 에 대해
    topics 리스트의 각 uorb 토픽을 DataFrame 으로 잘라서 반환.
    """
    trimmed = {}

    '''
    if bias_info and stab_time:
        start_us = bias_info['Time'] - int(10*1e6)
        end_us   = stab_time    + int(10*1e6)

        trimmed = trim_topics(ulog, bias_info, start_us, end_us)
    '''

    if bias_info:
        start_us = bias_info['Time'] - int(60 * 1e6)
        end_us = bias_info['Time'] + int(60 * 1e6)

        trimmed = trim_topics(ulog, start_us, end_us, topics)

    return trimmed

In [9]:
def trim_topics(ulog, start_us, end_us, topics):
    trimmed = {}

    for topic in topics:
        df = pd.DataFrame(ulog.get_dataset(topic).data)
        mask = (df['timestamp'] >= start_us) & (df['timestamp'] <= end_us)
        trimmed[topic] = df[mask].reset_index(drop=True)

    return trimmed

In [10]:
def process_test_case(args):
    base_log_dir, base_ulg_dir, iteration, model, test_case, topics = args

    try:
        # 1) combined log 읽기 & bias injection 파싱
        log_lines = get_log_data(base_log_dir, iteration, test_case, model)

        # 2) 착륙/추락 여부 파싱
        landing_outcome, bias_info, ulg_name, ekf_detection = parse_combined_log(log_lines)

        # 3) ULog 열기
        ulog = get_ulog(base_ulg_dir, ulg_name)

        if ulog:
            # 4) Arm 시점 추출
            arm_time = get_arm_time(ulog)

            # 4) stabilization time (bias injection 이후)
            stab_time = get_stab_time(ulog, bias_info)

            # 5) trim 구간 계산
            trimmed = get_trimmed_topics(ulog, bias_info, stab_time, topics)

            # 메모리 정리
            del ulog
            gc.collect()

            return iteration, test_case, {
                "bias_info": bias_info,
                "landing_outcome": landing_outcome,
                "ekf_detection": ekf_detection,
                "arm_time": arm_time,
                "stabilization_time": stab_time,
                "trimmed": trimmed
            }

        return iteration, test_case, {
            "bias_info": bias_info,
            "landing_outcome": landing_outcome,
            "ekf_detection": ekf_detection,
            "arm_time": None,
            "stabilization_time": None,
            "trimmed": {}
        }

    except Exception as e:
        return iteration, test_case, {"error": str(e)}

In [11]:
def load_all(base_log_dir, base_ulg_dir, topics, max_workers=None):
    if max_workers is None:
        max_workers = max(1, multiprocessing.cpu_count() // 2)

    tasks = []
    for itr in os.listdir(base_log_dir):
        for mdl in os.listdir(os.path.join(base_log_dir, itr)):
            for tc in os.listdir(os.path.join(base_log_dir, itr, mdl)):
                #if not tc.lower().startswith("normal"):
                #    continue
                tasks.append((base_log_dir, base_ulg_dir, itr, mdl, tc, topics))

    total_tasks = len(tasks)
    completed = 0
    results = {}

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_test_case, t): t for t in tasks}
        for fut in as_completed(futures):
            itr, tc, res = fut.result()
            results.setdefault(itr, {})[tc] = res
            completed += 1
            if completed % 10 == 0 or completed == total_tasks:
                print(f"Progress: {completed}/{total_tasks} ({completed / total_tasks * 100:.1f}%)")

    return results


In [12]:
base_log_dir = os.path.expanduser("~/ws/PX4-Autopilot/logs/2025-04-22T09-38-06Z")
base_ulg_dir = os.path.expanduser("~/ws/PX4-Autopilot/build/px4_sitl_default/tmp_mavsdk_tests/rootfs")
ulog_topics = [
    "vehicle_local_position_groundtruth",
    "vehicle_angular_velocity_groundtruth",
    "vehicle_attitude_groundtruth"
]

flight_data = load_all(base_log_dir, base_ulg_dir, ulog_topics)

Progress: 10/1200 (0.8%)
Progress: 20/1200 (1.7%)
Progress: 30/1200 (2.5%)
Progress: 40/1200 (3.3%)
Progress: 50/1200 (4.2%)
Progress: 60/1200 (5.0%)
Progress: 70/1200 (5.8%)
Progress: 80/1200 (6.7%)
Progress: 90/1200 (7.5%)
Progress: 100/1200 (8.3%)
Progress: 110/1200 (9.2%)
Progress: 120/1200 (10.0%)
Progress: 130/1200 (10.8%)
Progress: 140/1200 (11.7%)
Progress: 150/1200 (12.5%)
Progress: 160/1200 (13.3%)
Progress: 170/1200 (14.2%)
Progress: 180/1200 (15.0%)
Progress: 190/1200 (15.8%)
Progress: 200/1200 (16.7%)
Progress: 210/1200 (17.5%)
Progress: 220/1200 (18.3%)
Progress: 230/1200 (19.2%)
Progress: 240/1200 (20.0%)
Progress: 250/1200 (20.8%)
Progress: 260/1200 (21.7%)
Progress: 270/1200 (22.5%)
Progress: 280/1200 (23.3%)
Progress: 290/1200 (24.2%)
Progress: 300/1200 (25.0%)
Progress: 310/1200 (25.8%)
Progress: 320/1200 (26.7%)
Progress: 330/1200 (27.5%)
Progress: 340/1200 (28.3%)
Progress: 350/1200 (29.2%)
Progress: 360/1200 (30.0%)
Progress: 370/1200 (30.8%)
Progress: 380/1200 (3

In [20]:
import os


def plot_local_trajectories(flight_data, out_dir="images"):
    # 출력 디렉토리 생성
    os.makedirs(out_dir, exist_ok=True)

    """
    flight_data: dict
      iteration -> test_case -> info dict
    info dict contains:
      - bias_info: dict with 'Time'
      - trimmed: dict of DataFrames, includes 'vehicle_local_position_groundtruth'
    """
    # 그룹화: 동일 test_case 이름별로 모으기
    groups = defaultdict(list)
    for itr, cases in flight_data.items():
        for tc, info in cases.items():
            bias = info.get('bias_info')
            loc_df = info.get('trimmed', {}).get('vehicle_local_position_groundtruth')
            if bias and loc_df is not None and not loc_df.empty:
                groups[tc].append((itr, info))

    # 각 test_case마다 하나의 plot 생성
    for tc, runs in groups.items():
        fig, ax = plt.subplots(figsize=(10, 10))

        for itr, info in runs:
            df_loc = info['trimmed']['vehicle_local_position_groundtruth']
            bias_time = info['bias_info']['Time']

            # bias_time 이후의 첫 위치를 기준점으로 사용
            df_after = df_loc[df_loc['timestamp'] >= bias_time]
            if df_after.empty:
                df_after = df_loc
            base = df_after.iloc[0]
            x0, y0 = base['x'], base['y']

            # 좌표 이동
            dx = df_loc['x'] - x0
            dy = df_loc['y'] - y0

            # 경로 그리기
            plt.plot(dx, dy, label=f'Iter {itr}')

        # 원점 표시
        plt.scatter(0, 0, marker='x', s=50, label='Bias point')

        plt.title(f"Trajectory aligned at bias for {tc}")
        plt.xlabel('x (m)')
        plt.ylabel('y (m)')
        #plt.legend(loc='best')
        plt.grid(True)

        out_path = os.path.join(out_dir, f"{tc}.png")
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

In [21]:
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import matplotlib as mpl


def plot_local_trajectories_with_colormap(flight_data, out_dir="images"):
    # 출력 디렉토리 생성
    os.makedirs(out_dir, exist_ok=True)

    groups = defaultdict(list)
    for itr, cases in flight_data.items():
        for tc, info in cases.items():
            df = info.get('trimmed', {}) \
                .get('vehicle_local_position_groundtruth')
            if info.get('bias_info') and df is not None and not df.empty:
                groups[tc].append(info)

    for tc, runs in groups.items():
        fig, ax = plt.subplots(figsize=(10, 10))

        for info in runs:
            df = info['trimmed']['vehicle_local_position_groundtruth']
            bias = info['bias_info']['Time']

            # 기준점
            post = df[df['timestamp'] >= bias]
            base = post.iloc[0] if not post.empty else df.iloc[0]
            dx = df['x'] - base['x']
            dy = df['y'] - base['y']

            delta_us = df['timestamp'].astype(np.int64) - np.int64(bias)
            t = (delta_us // 1_000_000).astype(int)

            # discrete colormap
            '''
            seconds = np.unique(t)
            cmap = plt.colormaps['rainbow'].resampled(len(seconds))
            boundaries = np.concatenate([seconds - 0.5, [seconds[-1] + 0.5]])
            norm = mpl.colors.BoundaryNorm(boundaries, ncolors=len(seconds))
            '''

            t_bin5 = (t // 5) * 5
            bins = np.unique(t_bin5)

            # discrete colormap (rainbow)
            cmap = plt.colormaps['rainbow'].resampled(len(bins))
            bounds = np.concatenate([bins - 2.5, [bins[-1] + 2.5]])
            norm = mpl.colors.BoundaryNorm(bounds, ncolors=len(bins))

            ax.scatter(dx, dy, c=t, cmap=cmap, norm=norm,
                       s=5, alpha=0.8, edgecolors='none')

        ax.scatter(0, 0, marker='x', color='k', s=50)
        ax.set_title(f"{tc} – Trajectory aligned at bias")
        ax.set_xlabel('Δx (m)')
        ax.set_ylabel('Δy (m)')
        ax.axis('equal')
        ax.grid(True)

        sm = mpl.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, ticks=bins)
        cbar.ax.set_yticklabels([str(s) for s in bins])
        cbar.set_label('Time since bias (s)')
        out_path = os.path.join(out_dir, f"{tc}_colorMap.png")
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

In [22]:
plot_local_trajectories(flight_data)
plot_local_trajectories_with_colormap(flight_data)

In [23]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# quaternion → Euler (degrees) 변환 (벡터화)
def quaternion_to_euler_deg(qw, qx, qy, qz):
    t0 = 2*(qw*qx + qy*qz)
    t1 = 1 - 2*(qx*qx + qy*qy)
    roll  = np.arctan2(t0, t1)

    t2 = 2*(qw*qy - qz*qx)
    t2 = np.clip(t2, -1.0, 1.0)
    pitch = np.arcsin(t2)

    t3 = 2*(qw*qz + qx*qy)
    t4 = 1 - 2*(qy*qy + qz*qz)
    yaw   = np.arctan2(t3, t4)

    return roll*180/np.pi, pitch*180/np.pi, yaw*180/np.pi

# attitude runs 수집 (bias_info가 None이면 건너뜀)
def collect_attitude_runs(flight_data):
    groups = defaultdict(list)
    for itr, cases in flight_data.items():
        for tc, info in cases.items():
            bias_info = info.get('bias_info')
            if not bias_info or 'Time' not in bias_info:
                continue
            bias_time = bias_info['Time']
            df_att = info.get('trimmed', {}).get('vehicle_attitude_groundtruth')
            if df_att is None or df_att.empty:
                continue
            groups[tc].append((itr, bias_time, df_att.reset_index(drop=True)))
    return groups

# Roll 그리기
def plot_roll(flight_data, out_dir="images"):
    os.makedirs(out_dir, exist_ok=True)

    groups = collect_attitude_runs(flight_data)
    for tc, runs in groups.items():
        fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
        for itr, bias_time, df in runs:
            # x축: raw timestamp – bias_time (μs 단위)
            t = df['timestamp'].astype(np.int64) - np.int64(bias_time)
            # quaternion→Euler
            roll, _, _ = quaternion_to_euler_deg(
                df['q[0]'].to_numpy(),
                df['q[1]'].to_numpy(),
                df['q[2]'].to_numpy(),
                df['q[3]'].to_numpy()
            )
            ax.plot(t, roll, lw=0.8, label=f'Iter {itr}')

        ax.axvline(0, color='red', linestyle='--', label='Bias injection')
        ax.set_title(f"{tc} – Roll vs (timestamp – bias) [μs]")
        ax.set_xlabel('Time since bias (μs)')
        ax.set_ylabel('Roll (°)')
        ax.grid(True)

        out_path = os.path.join(out_dir, f"{tc}_roll.png")
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

# Pitch 그리기
def plot_pitch(flight_data, out_dir="images"):
    os.makedirs(out_dir, exist_ok=True)

    groups = collect_attitude_runs(flight_data)
    for tc, runs in groups.items():
        fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
        for itr, bias_time, df in runs:
            t = df['timestamp'].astype(np.int64) - np.int64(bias_time)
            _, pitch, _ = quaternion_to_euler_deg(
                df['q[0]'].to_numpy(),
                df['q[1]'].to_numpy(),
                df['q[2]'].to_numpy(),
                df['q[3]'].to_numpy()
            )
            ax.plot(t, pitch, lw=0.8, label=f'Iter {itr}')

        ax.axvline(0, color='red', linestyle='--', label='Bias injection')
        ax.set_title(f"{tc} – Pitch vs (timestamp – bias) [μs]")
        ax.set_xlabel('Time since bias (μs)')
        ax.set_ylabel('Pitch (°)')
        ax.grid(True)

        out_path = os.path.join(out_dir, f"{tc}_pitch.png")
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)

# Yaw 그리기
def plot_yaw(flight_data, out_dir="images"):
    os.makedirs(out_dir, exist_ok=True)

    groups = collect_attitude_runs(flight_data)
    for tc, runs in groups.items():
        fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)
        for itr, bias_time, df in runs:
            t = df['timestamp'].astype(np.int64) - np.int64(bias_time)
            _, _, yaw = quaternion_to_euler_deg(
                df['q[0]'].to_numpy(),
                df['q[1]'].to_numpy(),
                df['q[2]'].to_numpy(),
                df['q[3]'].to_numpy()
            )
            ax.plot(t, yaw, lw=0.8, label=f'Iter {itr}')

        ax.axvline(0, color='red', linestyle='--', label='Bias injection')
        ax.set_title(f"{tc} – Yaw vs (timestamp – bias) [μs]")
        ax.set_xlabel('Time since bias (μs)')
        ax.set_ylabel('Yaw (°)')
        ax.grid(True)

        out_path = os.path.join(out_dir, f"{tc}_yaw.png")
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)


In [24]:
plot_roll(flight_data)
plot_pitch(flight_data)
plot_yaw(flight_data)

In [25]:
plot_pitch(flight_data)

In [26]:
plot_yaw(flight_data)